In [7]:
!pip install -q sentence-transformers scikit-learn pandas pyarrow boto3

In [2]:
!df -h

Filesystem        Size  Used Avail Use% Mounted on
devtmpfs          4.0M     0  4.0M   0% /dev
tmpfs             1.9G     0  1.9G   0% /dev/shm
tmpfs             767M  8.7M  759M   2% /run
efivarfs          128K  2.7K  121K   3% /sys/firmware/efi/efivars
/dev/nvme0n1p1    105G   85G   21G  81% /
tmpfs             1.9G   97M  1.8G   6% /tmp
/dev/nvme0n1p128   10M  1.3M  8.7M  13% /boot/efi
/dev/nvme1n1       20G   48K   19G   1% /home/ec2-user/SageMaker
tmpfs             384M  4.0K  384M   1% /run/user/1000
tmpfs             384M  4.0K  384M   1% /run/user/1002
tmpfs             384M  4.0K  384M   1% /run/user/1001


In [3]:
!pip cache purge
!rm -rf ~/.cache/pip

Files removed: 1834 (6065.8 MB)
Directories removed: 33


In [6]:
!pip install -q sentence-transformers scikit-learn pandas pyarrow boto3


In [5]:
import os
os.environ['TMPDIR'] = '/home/ec2-user/SageMaker/tmp'
!mkdir -p /home/ec2-user/SageMaker/tmp
!pip install -q --no-cache-dir sentence-transformers scikit-learn pandas pyarrow boto3

In [8]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
print("All imports successful")

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [9]:
import pandas as pd

df = pd.read_parquet("s3://ravuri-kumar/complaints/")
df = df[df["narrative_word_count"] >= 5].reset_index(drop=True)
print(f"Loaded {len(df):,} complaints")
df.head(3)

Loaded 4,980 complaints


,complaint_id,date_received,product,sub_product,issue,company,state,narrative,company_response,timely_response,narrative_clean,narrative_word_count,year,month
0,6679424,2023-03-12,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,SYNCHRONY FINANCIAL,MA,Synchrony bank with XXXX reported a tradeline/...,Closed with explanation,Yes,Synchrony bank with [REDACTED] reported a trad...,102,2023,3
1,6679426,2023-03-12,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,"EQUIFAX, INC.",GA,The reporting agencies has failed to investiga...,Closed with explanation,Yes,The reporting agencies has failed to investiga...,122,2023,3
2,6679441,2023-03-12,"Credit reporting, credit repair services, or o...",Credit reporting,Improper use of your report,"Convergent Resources, Inc.",GA,In accordance with the fair credit Reporting a...,Closed with explanation,Yes,In accordance with the fair credit Reporting a...,80,2023,3


In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

# Sample for speed on this small instance
sample = df.sample(n=min(2000, len(df)), random_state=42).reset_index(drop=True)

embeddings = model.encode(
    sample["narrative_clean"].tolist(),
    batch_size=32,
    show_progress_bar=True,
)
embeddings = np.array(embeddings)
print(embeddings.shape)

Batches: 100%|██████████| 63/63 [02:00<00:00,  1.91s/it]

(2000, 384)


In [11]:
from sklearn.cluster import KMeans

N_CLUSTERS = 8
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
sample["topic_cluster_id"] = kmeans.fit_predict(embeddings)

sample.groupby("topic_cluster_id").size().sort_values(ascending=False)

topic_cluster_id
5    356
6    332
4    309
7    278
1    231
3    206
0    204
2     84
dtype: int64

In [12]:
for cid in range(8):
    print(f"\n--- Cluster {cid} ---")
    for txt in sample[sample.topic_cluster_id == cid]["narrative_clean"].head(2):
        print(" -", txt[:150], "...")


--- Cluster 0 ---
 - When I reviewed my credit report, I discovered that some of the information was erroneous. The 3 credit bureaus must validate these items in line with ...
 - I checked my credit report and found that some of the data wereincorrect. The [REDACTED] credit bureaus are required by Sections 609 ( a ) ( 1 ) ( A ) ...

--- Cluster 1 ---
 - In accordance with the Fair Credit Reporting act. The List of accounts below has violated my federally protected consumer rights to privacy and confid ...
 - In accordance with the Fair Credit Reporting act. The List of accounts below has violated my federally protected consumer rights to privacy and confid ...

--- Cluster 2 ---
 - My name is [REDACTED] [REDACTED] this complaint is not made in error neither is it being made by a third party.I declare under penalty of perjury I am ...
 - My name is [REDACTED] [REDACTED] this complaint is not made in error neither is it being made by a third party.I declare under penalty of perjury I am

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    embeddings, sample["issue"], test_size=0.2, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
print(classification_report(y_test, preds, zero_division=0))

                                                                                  precision    recall  f1-score   support

                         Credit monitoring or identity theft protection services       0.00      0.00      0.00         4
                                                     Improper use of your report       0.76      0.87      0.81       140
                                            Incorrect information on your report       0.68      0.61      0.65       135
                 Problem with a company's investigation into an existing problem       0.00      0.00      0.00         1
Problem with a credit reporting company's investigation into an existing problem       0.63      0.65      0.64       113
                                   Problem with fraud alerts or security freezes       0.00      0.00      0.00         1
                                Unable to get your credit report or credit score       0.00      0.00      0.00         5
                       

In [14]:
output_cols = ["complaint_id", "company", "product", "issue", "narrative_clean", "topic_cluster_id"]
sample[output_cols].to_parquet(
    "s3://ravuri-kumar/enriched/complaint_clusters.parquet",
    index=False,
)
print("Saved enriched cluster assignments to S3.")

Saved enriched cluster assignments to S3.


In [16]:
import boto3
import json

bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

response = bedrock.invoke_model(
    modelId="amazon.nova-micro-v1:0",
    body=json.dumps({
        "messages": [{"role": "user", "content": [{"text": "Say hello in one sentence."}]}],
        "inferenceConfig": {"maxTokens": 100}
    }),
    contentType="application/json",
    accept="application/json",
)
result = json.loads(response["body"].read())
print(result["output"]["message"]["content"][0]["text"])

Hello there! How are you today?


In [17]:
import boto3
import json
import pandas as pd

bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

df_enriched = pd.read_parquet("s3://ravuri-kumar/enriched/complaint_clusters.parquet")

# Pick a company that actually appears in your data (from earlier: EQUIFAX, INC. / TRANSUNION / etc.)
company = "EQUIFAX, INC."
narratives = df_enriched[df_enriched["company"] == company]["narrative_clean"].dropna().tolist()

prompt = f"""Summarize the most common issues in these customer complaints filed
against "{company}" in 3-4 sentences, in plain business language:

{chr(10).join(narratives[:15])}
"""

response = bedrock.invoke_model(
    modelId="amazon.nova-micro-v1:0",
    body=json.dumps({
        "messages": [{"role": "user", "content": [{"text": prompt}]}],
        "inferenceConfig": {"maxTokens": 400}
    }),
    contentType="application/json",
    accept="application/json",
)
result = json.loads(response["body"].read())
print(result["output"]["message"]["content"][0]["text"])

ServiceUnavailableException: An error occurred (ServiceUnavailableException) when calling the InvokeModel operation (reached max retries: 4): Service capacity limit has been reached. Please try again later.

In [18]:
import time
time.sleep(10)

response = bedrock.invoke_model(
    modelId="amazon.nova-micro-v1:0",
    body=json.dumps({
        "messages": [{"role": "user", "content": [{"text": prompt}]}],
        "inferenceConfig": {"maxTokens": 400}
    }),
    contentType="application/json",
    accept="application/json",
)
result = json.loads(response["body"].read())
print(result["output"]["message"]["content"][0]["text"])

The most common issues in these customer complaints against Equifax, Inc. revolve around inaccuracies and unauthorized items on credit reports, suspected identity theft, and breaches of privacy. Customers report seeing erroneous accounts and information that they did not authorize or were not informed about, which they believe violates their rights under the Fair Credit Reporting Act (FCRA). They are requesting validation, removal, and correction of these inaccuracies to restore the integrity of their credit reports.
